## naver_news 테이블의 한 행을 표현하기 위한 클래스 생성

In [1]:
# 네이버 뉴스 응답 데이터를 담을 NaverNews 엔티티 클래스 정의
from datetime import datetime

class NaverNews:

    def __init__(self, id: int, title: str, originallink: str, link: str, description: str, pub_date: str, created_at: datetime = None):
        self.__id = id
        self.__title = title
        self.__originallink = originallink
        self.__link = link
        self.__description = description
        self.__pub_date = pub_date
        self.__created_at = created_at

    @property
    def id(self):
        return self.__id

    @property
    def title(self):
        return self.__title

    # __title 속성에 대한 setter 메소드 예시
    # 아래 주석을 해제하면 naver_news.title = "새 제목"처럼 값을 변경할 수 있다.
    # @title.setter
    # def title(self, value):
    #     self.__title = value

    @property
    def originallink(self):
        return self.__originallink

    @property
    def link(self):
        return self.__link

    @property
    def description(self):
        return self.__description

    @property
    def pub_date(self):
        return self.__pub_date

    @property
    def created_at(self):
        return self.__created_at

    # naver news api 응답 데이터를 NaverNews 객체로 변환하기 위한 클래스 매서드
    @classmethod
    def from_api_item(cls, item:dict):
        return cls(
            id=None,
            title=item.get('title'),
            originallink=item.get('originallink'),
            link=item.get('link'),
            description=item.get('description'),
            pub_date=item.get('pubDate'),
        )   # 네이버 api의 pubDate와 변수명이 불일치해서 맞춰주는 작업
            # cls : 현재 클래스의 생성자(init)

    def __repr__(self):
        # 객체를 print() 했을 때 어떤 값이 들어 있는지 확인하기 위한 문자열 표현 메소드
        return f'NaverNews({self.__id}, {self.__title}, {self.__originallink}, {self.__link}, {self.__description}, {self.__pub_date}, {self.__created_at})'

## .env를 읽어서 환경변수로 등록

In [2]:
from dotenv import load_dotenv
import os

load_dotenv()   #.env를 읽어 환경변수로 등록

NAVER_CLIENT_ID = os.getenv("NAVER_CLIENT_ID")  #환경변수를 읽어 변수에 저장
NAVER_CLIENT_SECRET = os.getenv("NAVER_CLIENT_SECRET")

headers = {
    "X-Naver-Client-Id": NAVER_CLIENT_ID,
    "X-Naver-Client-Secret": NAVER_CLIENT_SECRET
}

    #기본값 적용하는 코드
config = {
    "host": os.getenv("DB_HOST", "localhost"), # (key, default)
    "port": os.getenv("DB_PORT", "3306"),  # 기본포트 3306인 경우 생략가능
    "user": os.getenv("DB_USER", "skn_ai"),
    "password": os.getenv("DB_PASSWORD", "1234"),
    "database": os.getenv("DB_DATABASE", "menudb")
}

if not NAVER_CLIENT_ID or not NAVER_CLIENT_SECRET:
    raise ValueError('NAVER_CLIENT_ID 또는 NAVER_CLIENT_SECRET이 설정되지 않았습니다.')

## 네이버 뉴스 "인공지능" 관련 최근 뉴스 10개를 조회해서 DB에 저장

In [3]:
import requests
from pprint import pprint   #출력 시 공백 문자를 이용해 가독성 좋게 출력

url = 'https://openapi.naver.com/v1/search/news.json'

params = {
    'query' : '인공지능',
    'display' : 10,
    'start' : 1,
    'sort' : 'date'
}

# 수집한 뉴스 데이터를 저장할 리스트
naver_news_list:list[NaverNews] = []

try:
    # GET Method : 조회 요청
    response = requests.get(
        url,
        headers=headers,
        params=params,  # dict -> 쿼리 스트링으로 변환(+ url encoding)
        timeout=10
    )

    # HTTP 상태 코드가 400, 500번 대인 경우 예외 발생
    response.raise_for_status()

    response_code = response.status_code    #상태코드
    data = response.json()                  #응답 데이터(json -> dict 변환)

    #pprint(data)
    #응답 데이터 중 뉴스 기사 리스트인 'items' 사용
    items = data.get('items',[])    # (key, default)

    #pprint(items)
    #-> list[dict]

    naver_news_list = [NaverNews.from_api_item(item) for item in items]

    print('response_code : ', response_code)
    print(naver_news_list)

except requests.exceptions.Timeout:
    print('요청 시간 10초 초과')
except ValueError:
    print('응답 데이터가 JSON 형식이 아닙니다')

response_code :  200
[NaverNews(None, 스페이스X 로켓 뜨면 삼성·SK하이닉스 반도체도 '난다', https://www.sisaweek.com/news/articleView.html?idxno=236122, https://www.sisaweek.com/news/articleView.html?idxno=236122, 또한 최근 도입되는 '<b>인공지능</b>(AI)' 기반 우주데이터센터도 마찬가지다. 따라서 우주 기반 AI인프라가 확대될수록 고성능 연산칩과 메모리 반도체의 중요성은 더욱 커질 수밖에 없다. 실제로 관련 시장 규모는 매해... , Mon, 15 Jun 2026 17:24:00 +0900, None), NaverNews(None, 한컴, 유럽 AI·R&amp;D 기업과 협력 강화…에이전틱 OS 현지화 추진, https://daily.hankooki.com/news/articleView.html?idxno=1376737, https://daily.hankooki.com/news/articleView.html?idxno=1376737, 사진=한컴 제공  한컴이 유럽 현지 <b>인공지능</b>(AI) 및 연구개발(R&amp;D) 기업들과 업무협약(MOU)을 체결하고 제품 공동 개발과 현지 시장 진출 기반 확보에 나선다. 한컴은 폴란드 국가공인 연구개발(R&amp;D) 센터 '7불스'와 MOU를... , Mon, 15 Jun 2026 17:24:00 +0900, None), NaverNews(None, 삼성전자, 글로벌 전략회의 돌입…폴더블·HBM·파운드리 로드맵 점검, https://www.newsway.co.kr/news/view?ud=2026061514305985362, https://www.newsway.co.kr/news/view?ud=2026061514305985362, 메모리 사업부는 <b>인공지능</b>(AI) 인프라 확대에 따른 수요 전망을 점검하고, 주요 고객사 대상 HBM 공급 현황을 면밀히 살필 것으로 보인

In [4]:
import mysql.connector

try:
    with mysql.connector.connect(**config) as conn:
        with conn.cursor() as cursor:
            for naver_news in naver_news_list:
                cursor.execute('''
                    INSERT INTO naver_news
                        (title, originallink, link, description, pub_date)
                    VALUES (%s,%s,%s,%s,%s)
                ''',(
                    naver_news.title,
                    naver_news.originallink,
                    naver_news.link,
                    naver_news.description,
                    naver_news.pub_date
                ))

            conn.commit()   # 전체 insert 내용 커밋

except mysql.connector.Error as err:
    print(f"DB 에러: {err}")
    conn.rollback()     #오류 발생 시 전체 roll back

In [5]:
# DB에 저장된 네이버 뉴스 데이터를 조회해 객체 리스트로 변환
try:
    with mysql.connector.connect(**config) as conn:
        with conn.cursor() as cursor:
            cursor.execute('''
                           select *
                           from naver_news
                           order by id desc
                           ''')  # sql, params
            naver_news_list = [NaverNews(*row) for row in cursor.fetchall()]
            print(naver_news_list)
except mysql.connector.Error as e:
    print('DB 오류:', e)

[NaverNews(42, 전남광주통합특별시 핵심 ‘기업유치·재정기획’ 두터운 라인업 구축, https://www.sedaily.com/article/20055972?ref=naver, https://n.news.naver.com/mnews/article/011/0004631324?sid=102, AI 및 경제 전략가로 꼽히는 임문영 국회의원(광주 광산구을·전 국가<b>인공지능</b>전략위원회 부위원장)이 AI·반도체 특별자문위원을 맡아 전략 수립의 전체 방향을 잡아갈 예정이다. 백승주 대전환기획위원회 부위원장은... , Mon, 15 Jun 2026 17:24:00 +0900, 2026-06-15 17:26:43), NaverNews(41, 심평원, 11개국 보건의료 전문가 대상 HIRA 국제연수과정 개최, https://www.akomnews.com/bbs/board.php?bo_table=news&wr_id=67461, https://www.akomnews.com/bbs/board.php?bo_table=news&wr_id=67461, 특히 올해는 세계보건기구 서태평양지역사무처 전문가 강의를 확대해 보편적 건강보장 달성을 위한 전략적 구매와 보건의료 재정 관리에 대한 교육을 강화했으며, 아울러 디지털 전환(DX), <b>인공지능</b>(AI), 보건의료... , Mon, 15 Jun 2026 17:24:00 +0900, 2026-06-15 17:26:43), NaverNews(40, 윤건영 충북교육감 당선인 '2기 공감동행교육출범준비위' 가동, http://www.dynews.co.kr/news/articleView.html?idxno=854241, http://www.dynews.co.kr/news/articleView.html?idxno=854241, 미래교육 디자인 분야 권위자인 오 교수는 지난 4년간 충북교육 성과를 진단하고, <b>인공지능</b>(AI) 시대에 대응할 새로운 4년의 충북 교육 정책을 고도화하는 역할을 맡는다. 위원회는 직속 기관인

## 중복 뉴스 방지
- 중복 뉴스(중복 행)를 방지하기 위해서는
  중복 기준이 되는 컬럼을 설정하고,
  컬럼 값이 중복이면 update가 수행되게 해야함

- 중복 판단을 위해 PK 또는 UNIQUE 제약 조건을 이용
    - -> 빠른 중복 판단 가능


- `replace` 대신 `on duplicate key update` 구문 사용
    - replace : delete -> insert
    - on duplicate key update : update 1회

```sql
ALTER TABLE naver_news
ADD CONSTRAINT uq_naver_news_link UNIQUE (link);
```

In [7]:
insert_count = 0
skip_count = 0

try:
    with mysql.connector.connect(**config) as conn:
        with conn.cursor() as cursor:
            for naver_news in naver_news_list:
                cursor.execute('''
                    insert into naver_news
                        (title, originallink, link, description, pub_date)
                    values
                        (%s, %s, %s, %s, %s)
                    on duplicate key update
                        link = link
                ''', (
                    naver_news.title,
                    naver_news.originallink,
                    naver_news.link,
                    naver_news.description,
                    naver_news.pub_date
                ))

                if cursor.rowcount == 1:
                    insert_count += 1
                else:
                    skip_count += 1

            conn.commit()

    print(f"신규 저장: {insert_count}건")
    print(f"중복 제외: {skip_count}건")

except mysql.connector.Error as e:
    print('DB 오류:', e)

신규 저장: 0건
중복 제외: 30건
